In [65]:
import pandas as pd
from pathlib import Path
folder = Path("../extract/arquivos_brutos/")

df_camp = pd.read_parquet(folder / "df_campeonatos.parquet")


In [66]:
#Tratando o titulo do df para ser o cabeçalho da planilha
df_camp.columns = df_camp.iloc[0]
df_camp = df_camp.iloc[1:].reset_index(drop=True)
df_camp.columns.name = None


In [67]:
df_camp.columns

Index(['Nome do Campeonato', 'Tier', 'STATUS', 'DAT. INICIO', 'DAT. TERMINO',
       'EMBED MATA-MATA', 'EMBED FASE DE GRUPOS', 'LINK', '1º Lugar',
       '2º Lugar', '3º Lugar', '4º Lugar', '5º Lugar', '6º Lugar', '7º Lugar',
       '8º Lugar'],
      dtype='str')

In [68]:
#tratando as colunas:
import re
import unicodedata
def normalizar_coluna(coluna):
  coluna = str(coluna).strip().lower()

  coluna = unicodedata.normalize("NFKD", coluna)

  coluna = "".join(
    c for c in coluna if not unicodedata.combining(c)
  )

  coluna = re.sub(r"[^a-z0-9]+", "_", coluna)

  return coluna.strip("_")

In [69]:
#Aplicando tratamento em todas as colunas
df_camp.columns = [
  normalizar_coluna(coluna) 
  for coluna in df_camp.columns
]
df_camp.columns

Index(['nome_do_campeonato', 'tier', 'status', 'dat_inicio', 'dat_termino',
       'embed_mata_mata', 'embed_fase_de_grupos', 'link', '1o_lugar',
       '2o_lugar', '3o_lugar', '4o_lugar', '5o_lugar', '6o_lugar', '7o_lugar',
       '8o_lugar'],
      dtype='str')

In [70]:
df_camp.head(3)

,nome_do_campeonato,tier,status,dat_inicio,dat_termino,embed_mata_mata,embed_fase_de_grupos,link,1o_lugar,2o_lugar,3o_lugar,4o_lugar,5o_lugar,6o_lugar,7o_lugar,8o_lugar
0,Campeonato Mineiro de Esportes Eletrônicos 2026,TIER S,EM ANDAMENTO,04/07/2026,25/07/2026,"<iframe src=""https://battlefy.com/embeds/6a399...","<iframe src=""https://battlefy.com/embeds/6a399...",https://battlefy.com/federa%C3%A7%C3%A3o-minei...,,,FEAR CLAN,GHOST GAMING,INATEL E-SPORTS,VERDONHA,OAB PARÁ DE MINAS,UNIFAL CIENTEC
1,Campeonato Mineiro de Esportes Eletrônicos 202...,TIER A,FINALIZADO,28/06/2026,28/06/2026,"<iframe src=""https://battlefy.com/embeds/6a399...",,https://battlefy.com/federa%C3%A7%C3%A3o-minei...,VERDONHA,,,,,,,
2,Campeonato Mineiro de Esportes Eletrônicos 202...,TIER A,FINALIZADO,27/06/2026,27/06/2026,"<iframe src=""https://battlefy.com/embeds/6a399...",,https://battlefy.com/federa%C3%A7%C3%A3o-minei...,INATEL E-SPORTS,GHOST GAMING,GEN MG,,,,,


In [71]:
df_camp = df_camp.drop(columns=['embed_mata_mata', 'embed_fase_de_grupos'])

In [72]:
#Normalização dos campos da tabela

df_camp['nome_do_campeonato'] = df_camp['nome_do_campeonato'].apply(normalizar_coluna)

df_camp['tier'] = [
    valor.lower().replace(" ", "_")
  for valor in df_camp['tier']
]
df_camp['status'] = [
    valor.lower().replace(" ", "_")
  for valor in df_camp['status']
]

df_camp['dat_inicio'] = pd.to_datetime(
  df_camp['dat_inicio'],
  format= "%d/%m/%Y",
  errors="raise"
)

df_camp['dat_termino'] = pd.to_datetime(
  df_camp['dat_termino'],
  format= "%d/%m/%Y",
  errors="raise"
)

for coluna in df_camp.iloc[:, 5:13]:
  df_camp[coluna] = [
    valor.lower().replace(" ", "_").replace("-", "_")
    for valor in df_camp[coluna]
  ]

df_camp.head(3)

,nome_do_campeonato,tier,status,dat_inicio,dat_termino,link,1o_lugar,2o_lugar,3o_lugar,4o_lugar,5o_lugar,6o_lugar,7o_lugar,8o_lugar
0,campeonato_mineiro_de_esportes_eletronicos_2026,tier_s,em_andamento,2026-07-04,2026-07-25,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,,,fear_clan,ghost_gaming,inatel_e_sports,verdonha,oab_pará_de_minas,UNIFAL CIENTEC
1,campeonato_mineiro_de_esportes_eletronicos_202...,tier_a,finalizado,2026-06-28,2026-06-28,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,verdonha,,,,,,,
2,campeonato_mineiro_de_esportes_eletronicos_202...,tier_a,finalizado,2026-06-27,2026-06-27,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,inatel_e_sports,ghost_gaming,gen_mg,,,,,


In [73]:
df_camp.insert(
  loc=0,
  column="id_camp",
  value=range(len(df_camp))
)

In [74]:
df_camp.columns

Index(['id_camp', 'nome_do_campeonato', 'tier', 'status', 'dat_inicio',
       'dat_termino', 'link', '1o_lugar', '2o_lugar', '3o_lugar', '4o_lugar',
       '5o_lugar', '6o_lugar', '7o_lugar', '8o_lugar'],
      dtype='str')

In [75]:
colunas_torneios = [
  'id_camp',
  'nome_do_campeonato',
  'tier',
  'status',
  'link',
  'dat_inicio',
  'dat_termino',
]

In [76]:
#Corrigindo o nome dos campeonatos que sao de 2019 mas estao como 2026
correcoes = {
  "campeonato_mineiro_de_esportes_eletronicos_2026_qualify_1":
        "campeonato_mineiro_de_esportes_eletronicos_2019_qualify_1",

    "campeonato_mineiro_de_esportes_eletronicos_2026_qualify_2":
        "campeonato_mineiro_de_esportes_eletronicos_2019_qualify_2"
}

registros_2019 = df_camp['dat_inicio'].dt.year.eq(2019)

df_camp.loc[registros_2019, 'nome_do_campeonato'] = (
  df_camp.loc[registros_2019, 'nome_do_campeonato'].replace(correcoes)
)


In [77]:
dim_torneios = df_camp[colunas_torneios].copy()
dim_torneios.head()

,id_camp,nome_do_campeonato,tier,status,link,dat_inicio,dat_termino
0,0,campeonato_mineiro_de_esportes_eletronicos_2026,tier_s,em_andamento,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,2026-07-04,2026-07-25
1,1,campeonato_mineiro_de_esportes_eletronicos_202...,tier_a,finalizado,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,2026-06-28,2026-06-28
2,2,campeonato_mineiro_de_esportes_eletronicos_202...,tier_a,finalizado,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,2026-06-27,2026-06-27
3,3,campeonato_mineiro_de_esportes_eletronicos_2025,tier_s,finalizado,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,2025-09-13,2025-11-15
4,4,campeonato_mineiro_de_esportes_eletronicos_202...,tier_a,finalizado,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,2025-09-20,2025-09-20


In [78]:
df_camp.head(1)

,id_camp,nome_do_campeonato,tier,status,dat_inicio,dat_termino,link,1o_lugar,2o_lugar,3o_lugar,4o_lugar,5o_lugar,6o_lugar,7o_lugar,8o_lugar
0,0,campeonato_mineiro_de_esportes_eletronicos_2026,tier_s,em_andamento,2026-07-04,2026-07-25,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,,,fear_clan,ghost_gaming,inatel_e_sports,verdonha,oab_pará_de_minas,UNIFAL CIENTEC


In [79]:
colocacoes = df_camp.melt(
  id_vars=["id_camp"], #mantem o id do campeonato, repetindo-o para cada colocacao
  value_vars=['1o_lugar', '2o_lugar','3o_lugar','4o_lugar','5o_lugar','6o_lugar','7o_lugar','8o_lugar'], # Colunas que serão transformadas em linhas da tabela
  var_name="placement_rank", #Nova coluna que receberá as colunas acima, como valores em linha
  value_name="team_name" #Nova coluna que receberá os valores das antigas colunas, no caso os times
)

In [80]:
colocacoes.head(8)

,id_camp,placement_rank,team_name
0,0,1o_lugar,
1,1,1o_lugar,verdonha
2,2,1o_lugar,inatel_e_sports
3,3,1o_lugar,fear_clan
4,4,1o_lugar,gen_mg
5,5,1o_lugar,mig
6,6,1o_lugar,inss_team
7,7,1o_lugar,fear_clan


In [81]:
colocacoes['placement_rank'] = [
  int(valor[0])
  for valor in colocacoes['placement_rank']
]

In [82]:
colocacoes

,id_camp,placement_rank,team_name
0,0,1,
1,1,1,verdonha
2,2,1,inatel_e_sports
3,3,1,fear_clan
4,4,1,gen_mg
...,...,...,...
267,29,8,TEAM JOLINHO
268,30,8,TEAM POTATO
269,31,8,TEAM SOLO BETIM
270,32,8,LEGENDS TEAM


In [83]:
colocacoes['team_name'] = [
  valor.lower().replace(" ", "_")
  for valor in colocacoes['team_name']
]
colocacoes

,id_camp,placement_rank,team_name
0,0,1,
1,1,1,verdonha
2,2,1,inatel_e_sports
3,3,1,fear_clan
4,4,1,gen_mg
...,...,...,...
267,29,8,team_jolinho
268,30,8,team_potato
269,31,8,team_solo_betim
270,32,8,legends_team


In [84]:
colocacoes_preenchidas = colocacoes[colocacoes["team_name"] != ""]
colocacoes_preenchidas

,id_camp,placement_rank,team_name
1,1,1,verdonha
2,2,1,inatel_e_sports
3,3,1,fear_clan
4,4,1,gen_mg
5,5,1,mig
...,...,...,...
267,29,8,team_jolinho
268,30,8,team_potato
269,31,8,team_solo_betim
270,32,8,legends_team


In [85]:
dim_torneios.head(2)

,id_camp,nome_do_campeonato,tier,status,link,dat_inicio,dat_termino
0,0,campeonato_mineiro_de_esportes_eletronicos_2026,tier_s,em_andamento,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,2026-07-04,2026-07-25
1,1,campeonato_mineiro_de_esportes_eletronicos_202...,tier_a,finalizado,https://battlefy.com/federa%c3%a7%c3%a3o_minei...,2026-06-28,2026-06-28


In [86]:
colocacoes_preenchidas.head(2)

,id_camp,placement_rank,team_name
1,1,1,verdonha
2,2,1,inatel_e_sports


In [87]:
with pd.option_context("display.max_colwidth", None):
    display(
        df_camp.loc[
            df_camp["id_camp"].isin([20, 21]),
            ["id_camp", "nome_do_campeonato", "dat_inicio"]
        ]
    )

,id_camp,nome_do_campeonato,dat_inicio
20,20,campeonato_mineiro_de_esportes_eletronicos_2019_qualify_2,2019-07-12
21,21,campeonato_mineiro_de_esportes_eletronicos_2019_qualify_1,2019-07-11


In [88]:
dim_torneios.to_excel('../transform/excel_transform/campeonatos/Campeonatos.xlsx', index=False)

colocacoes_preenchidas.to_excel('../transform/excel_transform/campeonatos/Colocacoes_Campeonatos.xlsx', index=False)